### batch data processing

In [1]:
from pathlib import Path
import re
import time
import numpy as np
import pandas as pd
import os
import mrcfile
import shutil
import re

In [3]:
def convert_one_xds_ascii(infile: Path, out_dir: Path) -> Path:
    s = str(infile)
    m = re.search(r'(?:^|[\\/])experiment[-_](\d+)(?:[\\/])', s, flags=re.IGNORECASE)
    if m:
        base = f"exp{m.group(1)}.hkl"
    else:
        p1 = infile.parent.name.replace(' ', '_')
        p2 = infile.parent.parent.name.replace(' ', '_') if infile.parent.parent else 'root'
        base = f"{p2}_{p1}.hkl"

    out_path = out_dir / base

    written = 0
    with infile.open('r', encoding='utf-8', errors='ignore') as fin, \
         out_path.open('w', encoding='utf-8') as fout:
        for line in fin:
            ls = line.lstrip()
            if not ls or ls.startswith('!'):
                continue  
            parts = ls.split()
            if len(parts) < 5:
                continue
            try:
                h, k, l = int(float(parts[0])), int(float(parts[1])), int(float(parts[2]))
                I, sigI = float(parts[3]), float(parts[4])
            except ValueError:
                continue
            fout.write(f"{h:4d} {k:4d} {l:4d} {I:12.2f} {sigI:12.2f}\n")
            written += 1

    if written == 0:
        try:
            out_path.unlink()
        except FileNotFoundError:
            pass
        raise RuntimeError(f"No valid data rows found in {infile}")
    return out_path

def batch_convert_xds_ascii(root: Path, out_dir: Path = None):
    if out_dir is None:
        out_dir = Path.cwd() / "exphkls"
    out_dir.mkdir(parents=True, exist_ok=True)

    files = list(root.rglob("XDS_ASCII.HKL"))
    if not files:
        print(f"⚠️ no XDS_ASCII.HKL under{root}  ")
        return []
    results = []
    print(f"📂 输入根目录: {root}")
    print(f"📦 输出目录  : {out_dir}\n")
    for f in files:
        try:
            print(f"⏳ 处理: {f}")
            outp = convert_one_xds_ascii(f, out_dir)
            print(f"   → 输出: {outp.name}")
            results.append((f, outp))
        except Exception as e:
            print(f"   ❌ 跳过（原因：{e})")
    print(f"\n🎉 完成！成功转换 {len(results)} 个文件。")
    return results
def parse_exp_num(name):
    """从 expN.hkl 提取 N"""
    m = re.match(r"exp(\d+)\.hkl$", name, flags=re.IGNORECASE)
    return int(m.group(1)) if m else None
    
def weighted_mean(values, sigmas):
    """加权平均 I；若 σI 不可用则普通平均"""
    values, sigmas = np.asarray(values, float), np.asarray(sigmas, float)
    mask = np.isfinite(values) & np.isfinite(sigmas) & (sigmas>0)
    if mask.sum() == 0:
        return np.nan
    w = 1.0 / (sigmas[mask]**2)
    return np.sum(w * values[mask]) / np.sum(w)
def extract_int():
    records = []
    for f in sorted(HKL_DIR.glob("exp*.hkl")):
        exp_num = parse_exp_num(f.name)
        if exp_num is None:
            continue
        df = pd.read_csv(f, delim_whitespace=True, header=None, names=["h","k","l","I","SIGI"])
        h,k,l = TARGET_HKL
        if MERGE_FRIEDEL:
            sel = ((df.h==h)&(df.k==k)&(df.l==l)) | ((df.h==-h)&(df.k==-k)&(df.l==-l))
        else:
            sel = (df.h==h)&(df.k==k)&(df.l==l)
        df_sel = df[sel]
        if not df_sel.empty:
            Imean = weighted_mean(df_sel["I"], df_sel["SIGI"])
            records.append((exp_num, Imean))
    
    if records:
        out_df = pd.DataFrame(sorted(records), columns=["expN","Intensity"])
        out_df.to_csv(outfile, sep="\t", index=False, header=False, float_format="%.4f")
        print(f"✅ 已生成文件: {outfile}")
        print(out_df.head())
    else:
        print(f"⚠️ 没有在 {HKL_DIR} 里找到反射 {TARGET_HKL} 的数据。")

In [5]:
def sum_intensity_in_mrc(path):
    """Return the sum of voxel intensities in an MRC file."""
    with mrcfile.open(path, permissive=True) as mrc:
        data = mrc.data.astype(np.float64)
        return float(np.sum(data))


def compute_dataset_sums(base_dir, inner_mrc_dir="RED"):
    """Compute summed intensity for every experiment_x."""
    results = {}

    # --- 关键改动：自然数顺序排序 experiment ---
    def experiment_key(name):
        m = re.search(r'experiment_(\d+)', name)
        return int(m.group(1)) if m else float('inf')

    for exp_name in sorted(os.listdir(base_dir), key=experiment_key):
        exp_path = os.path.join(base_dir, exp_name)
        if not (os.path.isdir(exp_path) and exp_name.startswith("experiment_")):
            continue

        mrc_dir = os.path.join(exp_path, inner_mrc_dir)
        if not os.path.isdir(mrc_dir):
            print(f"Skip {exp_name}: no '{inner_mrc_dir}' folder.")
            continue

        mrc_files = [f for f in os.listdir(mrc_dir) if f.lower().endswith(".mrc")]
        if not mrc_files:
            print(f"Skip {exp_name}: no .mrc files.")
            continue

        print(f"Processing {exp_name}...")

        total_sum = 0.0
        for fname in mrc_files:
            total_sum += sum_intensity_in_mrc(os.path.join(mrc_dir, fname))

        results[exp_name] = total_sum
        print(f"  → sum = {total_sum}")

    return results


def save_results_to_txt(results, output_path):
    """Save sums without scientific notation."""
    with open(output_path, "w") as f:
        f.write("Dataset\tTotal_Intensity\n")
        for exp, val in results.items():
            f.write(f"{exp}\t{val:.0f}\n")

    print(f"\nSaved results to: {output_path}")

In [ ]:
#normalization number
BASE_DIR = r"C:\Users\luoyi\Desktop\Data\PdSe2\20260317_1ps"
INNER_MRC_DIR = "RED"
OUTPUT_TXT = os.path.join(BASE_DIR, "intensity_sums.txt")

if __name__ == "__main__":
    sums = compute_dataset_sums(BASE_DIR, INNER_MRC_DIR)
    save_results_to_txt(sums, OUTPUT_TXT)

    print("\n=== SUMMARY ===")
    for exp, val in sums.items():
        print(f"{exp}: {val:.6e}")


### extract intensities

In [ ]:
ROOT_DIR = r'C:\Users\luoyi\Desktop\Data\PdSe2\20260317_1ps'

results = batch_convert_xds_ascii(Path(ROOT_DIR), Path(ROOT_DIR))

In [ ]:
HKL_DIR = Path(r"C:\Users\luoyi\Desktop\Data\PdSe2\20260317_1ps\intensities")      
TARGET_HKL = (0, -2, 0) 
MERGE_FRIEDEL = False
outfile = f"{TARGET_HKL[0]}{TARGET_HKL[1]}{TARGET_HKL[2]}.txt"

extract_int()

### extract .P4P & .HKL files

In [ ]:
# 你的总目录
root = Path(r"C:\Users\luoyi\Desktop\Data\PdSe2\20260317_1ps")

# 输出目录
out_root = root / "ss"
out_root.mkdir(exist_ok=True)

# 匹配 experiment_N
pattern = re.compile(r"experiment_(\d+)$")

# 遍历所有 experiment_N 文件夹
for exp_dir in root.iterdir():
    if not exp_dir.is_dir():
        continue

    m = pattern.match(exp_dir.name)
    if not m:
        continue

    n = m.group(1)  # 提取 N
    xds_dir = exp_dir / "SMV" / "data" / "xds"

    p4p_src = xds_dir / "1.P4P"
    hkl_src = xds_dir / "1.HKL"

    # 如果两个文件都不存在，就跳过
    if not p4p_src.exists() and not hkl_src.exists():
        print(f"跳过 {exp_dir.name}: 没找到 1.P4P 或 1.HKL")
        continue

    # 创建目标文件夹 ss\expN
    exp_out = out_root / f"exp{n}"
    exp_out.mkdir(parents=True, exist_ok=True)

    # 复制并重命名
    if p4p_src.exists():
        p4p_dst = exp_out / f"{n}.P4P"
        shutil.copy2(p4p_src, p4p_dst)
        print(f"已复制: {p4p_src} -> {p4p_dst}")
    else:
        print(f"{exp_dir.name} 中未找到 1.P4P")

    if hkl_src.exists():
        hkl_dst = exp_out / f"{n}.HKL"
        shutil.copy2(hkl_src, hkl_dst)
        print(f"已复制: {hkl_src} -> {hkl_dst}")
    else:
        print(f"{exp_dir.name} 中未找到 1.HKL")

print("完成。")

### extract atom column

In [7]:
root_folder = r"C:\Users\luoyi\Desktop\Data\PdSe2\20251218\solution\sg2\cif_Pbca"   
output_long_txt = "atom_positions_long.txt"
output_wide_txt = "atom_positions_wide.txt"

In [9]:
def parse_cif_number_with_uncertainty(val):
    """
    解析 CIF 数值并保留不确定度

    例如:
    0.1103(5) -> raw='0.1103(5)', value=0.1103, uncertainty=0.0005
    1.0       -> raw='1.0', value=1.0, uncertainty=None
    """
    if val is None:
        return {'raw': None, 'value': None, 'uncertainty': None}

    s = str(val).strip().strip("'").strip('"')

    if s in ['?', '.']:
        return {'raw': s, 'value': None, 'uncertainty': None}

    m = re.fullmatch(r'([+-]?\d*\.?\d+)(?:\((\d+)\))?', s)
    if not m:
        return {'raw': s, 'value': None, 'uncertainty': None}

    value_str = m.group(1)
    esd_digits = m.group(2)

    value = float(value_str)
    uncertainty = None

    if esd_digits is not None:
        if '.' in value_str:
            decimals = len(value_str.split('.')[-1])
        else:
            decimals = 0
        uncertainty = int(esd_digits) * (10 ** (-decimals))

    return {'raw': s, 'value': value, 'uncertainty': uncertainty}


def split_cif_line(line):
    """
    按空白分割，同时保留引号中的整体字符串
    """
    return re.findall(r"(?:'[^']*'|\"[^\"]*\"|\S+)", line.strip())


In [11]:
def extract_atom_sites_from_cif(cif_path, root_folder):
    with open(cif_path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    records = []
    i = 0
    n = len(lines)

    while i < n:
        line = lines[i].strip()

        if line == 'loop_':
            i += 1
            headers = []

            while i < n and lines[i].strip().startswith('_'):
                headers.append(lines[i].strip())
                i += 1

            # 只抓 atom_site 主表，不抓 aniso 表
            if headers and any(h.startswith('_atom_site_') for h in headers) \
               and not any(h.startswith('_atom_site_aniso_') for h in headers):

                while i < n:
                    s = lines[i].strip()

                    if (not s or
                        s == 'loop_' or
                        s.startswith('_') or
                        s.startswith('data_')):
                        break

                    parts = split_cif_line(s)
                    if len(parts) >= len(headers):
                        row = dict(zip(headers, parts[:len(headers)]))
                        records.append(row)

                    i += 1
                continue

        i += 1

    # 对每个元素按出现顺序重新编号：Pd_1, Se_1, Se_2 ...
    element_counter = {}
    cleaned = []

    cif_path_obj = Path(cif_path)
    root_path_obj = Path(root_folder).resolve()

    for r in records:
        element = r.get('_atom_site_type_symbol')
        if element is None:
            continue

        element = str(element).strip().strip("'").strip('"')

        element_counter[element] = element_counter.get(element, 0) + 1
        element_index = element_counter[element]
        element_site = f"{element}_{element_index}"

        x = parse_cif_number_with_uncertainty(r.get('_atom_site_fract_x'))
        y = parse_cif_number_with_uncertainty(r.get('_atom_site_fract_y'))
        z = parse_cif_number_with_uncertainty(r.get('_atom_site_fract_z'))

        cleaned.append({
            'file_name': cif_path_obj.name,
            'relative_path': str(cif_path_obj.resolve().relative_to(root_path_obj)),

            'element': element,
            'element_index': element_index,
            'element_site': element_site,

            'x_raw': x['raw'],
            'x': x['value'],
            'x_unc': x['uncertainty'],

            'y_raw': y['raw'],
            'y': y['value'],
            'y_unc': y['uncertainty'],

            'z_raw': z['raw'],
            'z': z['value'],
            'z_unc': z['uncertainty'],
        })

    return cleaned


In [13]:
def extract_all_cifs_recursive(root_folder, output_long_txt, output_wide_txt):
    root = Path(root_folder)

    def natural_sort_key(path_obj):
        """
        对路径做自然排序，例如：
        exp_2.cif < exp_10.cif
        sub3/file_2.cif < sub12/file_1.cif
        """
        s = str(path_obj.relative_to(root)).lower()
        return [int(text) if text.isdigit() else text for text in re.split(r'(\d+)', s)]

    cif_files = sorted(root.rglob("*.cif"), key=natural_sort_key)

    if not cif_files:
        print("没有找到任何 .cif 文件。")
        return None, None

    print(f"共找到 {len(cif_files)} 个 .cif 文件。\n")

    all_data = []
    for cif_file in cif_files:
        try:
            data = extract_atom_sites_from_cif(cif_file, root_folder)
            all_data.extend(data)
            print(f"Processed: {cif_file}")
        except Exception as e:
            print(f"Failed: {cif_file} -> {e}")

    df_long = pd.DataFrame(all_data)

    if df_long.empty:
        print("\n没有提取到 atom_site 数据。")
        return df_long, None

    # 保持文件顺序与读取顺序一致
    file_order = [str(f.resolve().relative_to(root.resolve())) for f in cif_files]
    df_long["relative_path"] = pd.Categorical(
        df_long["relative_path"],
        categories=file_order,
        ordered=True
    )

    # 文件顺序按自然排序，文件内部按元素和编号排序
    df_long = df_long.sort_values(
        by=['relative_path', 'element', 'element_index'],
        ascending=[True, True, True]
    ).reset_index(drop=True)

    # 保存长表
    long_out_path = root / output_long_txt
    df_long.to_csv(long_out_path, sep='\t', index=False, encoding='utf-8-sig')

    # 生成宽表
    wide_cols = ['x_raw', 'x', 'x_unc', 'y_raw', 'y', 'y_unc', 'z_raw', 'z', 'z_unc']

    df_wide = df_long.pivot(
        index=['file_name', 'relative_path'],
        columns='element_site',
        values=wide_cols
    )

    df_wide.columns = [f"{site}_{field}" for field, site in df_wide.columns]
    df_wide = df_wide.reset_index()

    # 宽表也按同样文件顺序排列
    df_wide["relative_path"] = pd.Categorical(
        df_wide["relative_path"],
        categories=file_order,
        ordered=True
    )
    df_wide = df_wide.sort_values("relative_path").reset_index(drop=True)

    # 保存宽表
    wide_out_path = root / output_wide_txt
    df_wide.to_csv(wide_out_path, sep='\t', index=False, encoding='utf-8-sig')

    print("\n完成。")
    print(f"长表已保存到: {long_out_path}")
    print(f"宽表已保存到: {wide_out_path}")

    return df_long, df_wide

In [15]:
df_long, df_wide = extract_all_cifs_recursive(
    root_folder=root_folder,
    output_long_txt=output_long_txt,
    output_wide_txt=output_wide_txt
)

print("\n===== 长表预览 =====")
display(df_long.head(20))

print("\n===== 宽表预览 =====")
display(df_wide.head(10))

共找到 19 个 .cif 文件。

Processed: C:\Users\luoyi\Desktop\Data\PdSe2\20251218\solution\sg2\cif_Pbca\2_1.cif
Processed: C:\Users\luoyi\Desktop\Data\PdSe2\20251218\solution\sg2\cif_Pbca\3_1.cif
Processed: C:\Users\luoyi\Desktop\Data\PdSe2\20251218\solution\sg2\cif_Pbca\4_1.cif
Processed: C:\Users\luoyi\Desktop\Data\PdSe2\20251218\solution\sg2\cif_Pbca\5_1.cif
Processed: C:\Users\luoyi\Desktop\Data\PdSe2\20251218\solution\sg2\cif_Pbca\6_1.cif
Processed: C:\Users\luoyi\Desktop\Data\PdSe2\20251218\solution\sg2\cif_Pbca\8_1.cif
Processed: C:\Users\luoyi\Desktop\Data\PdSe2\20251218\solution\sg2\cif_Pbca\9_1.cif
Processed: C:\Users\luoyi\Desktop\Data\PdSe2\20251218\solution\sg2\cif_Pbca\10_1.cif
Processed: C:\Users\luoyi\Desktop\Data\PdSe2\20251218\solution\sg2\cif_Pbca\11_1.cif
Processed: C:\Users\luoyi\Desktop\Data\PdSe2\20251218\solution\sg2\cif_Pbca\12_1.cif
Processed: C:\Users\luoyi\Desktop\Data\PdSe2\20251218\solution\sg2\cif_Pbca\13_1.cif
Processed: C:\Users\luoyi\Desktop\Data\PdSe2\20251218

,file_name,relative_path,element,element_index,element_site,x_raw,x,x_unc,y_raw,y,y_unc,z_raw,z,z_unc
0,2_1.cif,2_1.cif,Pd,1,Pd_1,0.000000,0.0000,NaN,0.000000,0.0000,NaN,0.000000,0.0000,NaN
1,2_1.cif,2_1.cif,Se,1,Se_1,0.1105(5),0.1105,0.0005,0.3823(4),0.3823,0.0004,0.0934(7),0.0934,0.0007
2,3_1.cif,3_1.cif,Pd,1,Pd_1,0.000000,0.0000,NaN,0.000000,0.0000,NaN,0.000000,0.0000,NaN
3,3_1.cif,3_1.cif,Se,1,Se_1,0.1107(5),0.1107,0.0005,0.3831(4),0.3831,0.0004,0.0942(7),0.0942,0.0007
4,4_1.cif,4_1.cif,Pd,1,Pd_1,0.000000,0.0000,NaN,0.000000,0.0000,NaN,0.000000,0.0000,NaN
5,4_1.cif,4_1.cif,Se,1,Se_1,0.1105(5),0.1105,0.0005,0.3829(5),0.3829,0.0005,0.0945(8),0.0945,0.0008
6,5_1.cif,5_1.cif,Pd,1,Pd_1,0.000000,0.0000,NaN,0.000000,0.0000,NaN,0.000000,0.0000,NaN
7,5_1.cif,5_1.cif,Se,1,Se_1,0.1103(5),0.1103,0.0005,0.3827(4),0.3827,0.0004,0.0937(8),0.0937,0.0008
8,6_1.cif,6_1.cif,Pd,1,Pd_1,0.000000,0.0000,NaN,0.000000,0.0000,NaN,0.000000,0.0000,NaN
9,6_1.cif,6_1.cif,Se,1,Se_1,0.1099(5),0.1099,0.0005,0.3833(4),0.3833,0.0004,0.0943(7),0.0943,0.0007



===== 宽表预览 =====


,file_name,relative_path,Pd_1_x_raw,Se_1_x_raw,Pd_1_x,Se_1_x,Pd_1_x_unc,Se_1_x_unc,Pd_1_y_raw,Se_1_y_raw,Pd_1_y,Se_1_y,Pd_1_y_unc,Se_1_y_unc,Pd_1_z_raw,Se_1_z_raw,Pd_1_z,Se_1_z,Pd_1_z_unc,Se_1_z_unc
0,2_1.cif,2_1.cif,0.000000,0.1105(5),0.0,0.1105,NaN,0.0005,0.000000,0.3823(4),0.0,0.3823,NaN,0.0004,0.000000,0.0934(7),0.0,0.0934,NaN,0.0007
1,3_1.cif,3_1.cif,0.000000,0.1107(5),0.0,0.1107,NaN,0.0005,0.000000,0.3831(4),0.0,0.3831,NaN,0.0004,0.000000,0.0942(7),0.0,0.0942,NaN,0.0007
2,4_1.cif,4_1.cif,0.000000,0.1105(5),0.0,0.1105,NaN,0.0005,0.000000,0.3829(5),0.0,0.3829,NaN,0.0005,0.000000,0.0945(8),0.0,0.0945,NaN,0.0008
3,5_1.cif,5_1.cif,0.000000,0.1103(5),0.0,0.1103,NaN,0.0005,0.000000,0.3827(4),0.0,0.3827,NaN,0.0004,0.000000,0.0937(8),0.0,0.0937,NaN,0.0008
4,6_1.cif,6_1.cif,0.000000,0.1099(5),0.0,0.1099,NaN,0.0005,0.000000,0.3833(4),0.0,0.3833,NaN,0.0004,0.000000,0.0943(7),0.0,0.0943,NaN,0.0007
5,8_1.cif,8_1.cif,0.000000,0.1103(7),0.0,0.1103,NaN,0.0007,0.000000,0.3835(5),0.0,0.3835,NaN,0.0005,0.000000,0.0949(8),0.0,0.0949,NaN,0.0008
6,9_1.cif,9_1.cif,0.000000,0.1102(5),0.0,0.1102,NaN,0.0005,0.000000,0.3831(4),0.0,0.3831,NaN,0.0004,0.000000,0.0955(8),0.0,0.0955,NaN,0.0008
7,10_1.cif,10_1.cif,0.000000,0.1100(5),0.0,0.11,NaN,0.0005,0.000000,0.3829(5),0.0,0.3829,NaN,0.0005,0.000000,0.0953(8),0.0,0.0953,NaN,0.0008
8,11_1.cif,11_1.cif,0.000000,0.1101(5),0.0,0.1101,NaN,0.0005,0.000000,0.3834(5),0.0,0.3834,NaN,0.0005,0.000000,0.0953(8),0.0,0.0953,NaN,0.0008
9,12_1.cif,12_1.cif,0.000000,0.1100(5),0.0,0.11,NaN,0.0005,0.000000,0.3827(5),0.0,0.3827,NaN,0.0005,0.000000,0.0951(8),0.0,0.0951,NaN,0.0008
